תהליך הרצת המודלים
שלב 1 – Baseline
Linear Regression – פשוט, מהיר, נותן נקודת התחלה
שלב 2 – מודלים מתקדמים
Random Forest
XGBoost
LightGBM
שלב 3 – השוואה על Validation
RMSE – Root Mean Squared Error
MAE – Mean Absolute Error
R² – מקדם הדטרמינציה
שלב 4 – Hyperparameter Tuning
על המודל הטוב ביותר בלבד
שלב 5 – הערכה סופית על Test
פעם אחת בלבד!

מדדי הצלחה:
מדדמשמעותRMSEכמה דקות טועים בממוצע (רגיש ל-outliers)MAEכמה דקות טועים בממוצע (פחות רגיש)R²כמה אחוז מהשונות מוסבר

סדר העבודה:
python# לכל מודל:
1. הגדר מודל
2. fit על X_train, y_train
3. predict על X_val
4. חשב RMSE, MAE, R²
5. השווה תוצאות

# imports

In [8]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [11]:
import os
print(os.getcwd())

C:\Users\Tamir\OneDrive - אליה הנדסה בע מ 100514861\Desktop\Tamir\pythonProj\public_transport_ML


# Load Data

In [12]:
# Load data
train_df = pd.read_csv(r'data/model_datasets/train_df_to_ml.csv', encoding='utf-8-sig')
val_df   = pd.read_csv(r'data/model_datasets/val_df_to_ml.csv', encoding='utf-8-sig')
test_df  = pd.read_csv(r'data/model_datasets/test_df_to_ml.csv', encoding='utf-8-sig')

print(f"Train:      {train_df.shape}")
print(f"Validation: {val_df.shape}")
print(f"Test:       {test_df.shape}")

Train:      (63366, 33)
Validation: (13590, 33)
Test:       (13589, 33)


# 1. Baseline - Linear Regression

In [13]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# הגדר X ו-y
target = 'duration_difference_min'

X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_val = val_df.drop(columns=[target])
y_val = val_df[target]

# === Baseline 1 - Dummy Regressor ===
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_val)

rmse_dummy = np.sqrt(mean_squared_error(y_val, y_pred_dummy))
mae_dummy  = mean_absolute_error(y_val, y_pred_dummy)
r2_dummy   = r2_score(y_val, y_pred_dummy)

print("=== Baseline 1 - Dummy Regressor (Mean) ===")
print(f"RMSE: {rmse_dummy:.4f}")
print(f"MAE:  {mae_dummy:.4f}")
print(f"R²:   {r2_dummy:.4f}")

# === Baseline 2 - Linear Regression ===
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_val)

rmse_lr = np.sqrt(mean_squared_error(y_val, y_pred_lr))
mae_lr  = mean_absolute_error(y_val, y_pred_lr)
r2_lr   = r2_score(y_val, y_pred_lr)

print("\n=== Baseline 2 - Linear Regression ===")
print(f"RMSE: {rmse_lr:.4f}")
print(f"MAE:  {mae_lr:.4f}")
print(f"R²:   {r2_lr:.4f}")

# === השוואה ===
print("\n=== Summary ===")
print(f"{'Model':<25} {'RMSE':>8} {'MAE':>8} {'R²':>8}")
print(f"{'Dummy (Mean)':<25} {rmse_dummy:>8.4f} {mae_dummy:>8.4f} {r2_dummy:>8.4f}")
print(f"{'Linear Regression':<25} {rmse_lr:>8.4f} {mae_lr:>8.4f} {r2_lr:>8.4f}")

=== Baseline 1 - Dummy Regressor (Mean) ===
RMSE: 18.1804
MAE:  12.8541
R²:   -0.0001

=== Baseline 2 - Linear Regression ===
RMSE: 0.0613
MAE:  0.0044
R²:   1.0000

=== Summary ===
Model                         RMSE      MAE       R²
Dummy (Mean)               18.1804  12.8541  -0.0001
Linear Regression           0.0613   0.0044   1.0000


In [16]:
# בדוק קורלציה מלאה
corr_full = X_train.corrwith(y_train).abs().sort_values(ascending=False)
print(corr_full)

duration_min_actual            0.623646
speed_kmh_actual               0.407249
Avg_Passengers_Per_Bus         0.376014
passengers_x_peak              0.299781
Total_Passengers               0.297035
stops_x_passengers             0.287975
is_peak_hour                   0.287712
origin_station_encoded         0.272463
destination_station_encoded    0.251281
perc_within_pt_route_peak      0.221071
origin_city_encoded            0.203490
destination_city_encoded       0.181719
agency_name_encoded            0.139210
urban                          0.134753
speed_kmh_planned              0.127824
arrival_hour                   0.127494
alternative_0                  0.118139
length_in_buffer_m             0.108978
route_length                   0.106671
number_of_stops                0.105190
route_length_km                0.103800
alternative_#                  0.087981
duration_min_planned           0.077548
full_hour                      0.065439
departure_hour                 0.065439


In [17]:
# בדוק אם duration_min_actual - duration_min_planned = duration_difference_min
diff = (train_df['duration_min_actual'] - train_df['duration_min_planned'] - train_df['duration_difference_min']).abs()
print(f"Max difference: {diff.max()}")
print(f"Mean difference: {diff.mean()}")
print(f"Rows where diff == 0: {(diff == 0).sum()} out of {len(train_df)}")

Max difference: 3.0
Mean difference: 0.0025876127050265306
Rows where diff == 0: 55965 out of 63366
